In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

import numpy as np
import pandas as pd

from sklearn import datasets
from sklearn.model_selection import train_test_split

# Tai du lieu Iris
iris = datasets.load_iris()

# Xem mo ta du lieu
print(iris.DESCR)

# Tach dac trung va nhan
X = iris.data
y = iris.target

print("Kich thuoc X:", X.shape)
print("Kich thuoc y:", y.shape)
print("Ten dac trung:", iris.feature_names)
print("Ten cac lop:", iris.target_names)


In [ ]:

# Chia du lieu thanh tap huan luyen va kiem tra
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=101
)

# Kiem tra kich thuoc cac tap du lieu
print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)


In [ ]:

from sklearn.preprocessing import StandardScaler

# Khoi tao bo chuan hoa
scaler = StandardScaler()

# Hoc tham so chuan hoa tu tap train
X_train_scaled = scaler.fit_transform(X_train)

# Ap dung cung tham so cho tap test
X_test_scaled = scaler.transform(X_test)

print("Du lieu train sau chuan hoa:")
print(X_train_scaled[:5])

print("Kich thuoc train:", X_train_scaled.shape)
print("Kich thuoc test:", X_test_scaled.shape)


In [ ]:

from sklearn import svm

# Khoi tao mo hinh SVM
clf = svm.SVC()

# Huan luyen mo hinh
clf.fit(X_train_scaled, y_train)

print("Huan luyen SVM thanh cong!")


In [ ]:

# Do chinh xac tren tap train
train_acc = clf.score(X_train_scaled, y_train)

# Do chinh xac tren tap test
val_acc = clf.score(X_test_scaled, y_test)

print("Training accuracy:", train_acc)
print("Validation accuracy:", val_acc)

print(f"Training accuracy: {train_acc * 100:.2f}%")
print(f"Validation accuracy: {val_acc * 100:.2f}%")


In [ ]:

from sklearn import svm
import pandas as pd

# Danh sach cac kernel can thu nghiem
kernels = ['linear', 'poly', 'rbf', 'sigmoid']

best_svm = None
best_val_acc = -1
best_kernel = None

results = []

# Thu nghiem tung kernel
for kernel in kernels:
    model = svm.SVC(kernel=kernel, C=1.0)

    # Huan luyen mo hinh
    model.fit(X_train_scaled, y_train)

    # Danh gia tren train va test
    train_acc = model.score(X_train_scaled, y_train)
    val_acc = model.score(X_test_scaled, y_test)

    results.append({
        'Kernel': kernel,
        'Train Accuracy': train_acc,
        'Test Accuracy': val_acc
    })

    # Luu ket qua cao nhat
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_svm = model
        best_kernel = kernel

# Hien thi bang ket qua
df_results = pd.DataFrame(results)
display(df_results)

print("Best Kernel:", best_kernel)
print(f"Best Accuracy: {best_val_acc * 100:.2f}%")


In [ ]:

import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))

plt.bar(
    df_results['Kernel'],
    df_results['Test Accuracy'] * 100
)

plt.xlabel("Kernel")
plt.ylabel("Accuracy (%)")
plt.title("Comparison of SVM Kernels - Iris")
plt.ylim(0, 105)

for i, acc in enumerate(df_results['Test Accuracy']):
    plt.text(
        i,
        acc * 100 + 1,
        f"{acc * 100:.2f}%",
        ha='center'
    )

plt.tight_layout()

plt.savefig(
    "/kaggle/working/iris_kernel_comparison.png",
    dpi=200
)

plt.show()


In [ ]:

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV, StratifiedKFold

# Tao pipeline chuan hoa va SVM
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('svm', SVC())
])

# Cac bo tham so can thu nghiem
param_grid = [
    {
        'svm__kernel': ['linear'],
        'svm__C': [0.1, 1, 10, 100]
    },
    {
        'svm__kernel': ['rbf'],
        'svm__C': [0.1, 1, 10, 100],
        'svm__gamma': ['scale', 0.01, 0.1, 1]
    },
    {
        'svm__kernel': ['poly'],
        'svm__C': [0.1, 1, 10],
        'svm__degree': [2, 3]
    },
    {
        'svm__kernel': ['sigmoid'],
        'svm__C': [0.1, 1, 10]
    }
]

# Chia 5 fold de danh gia
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=101
)

# Tim tham so tot nhat
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring='accuracy',
    n_jobs=-1,
    refit=True
)

# Huan luyen tren tap train goc
grid_search.fit(X_train, y_train)

print("Best parameters:", grid_search.best_params_)
print("Best CV accuracy:", grid_search.best_score_)


In [ ]:

from sklearn.metrics import accuracy_score, classification_report

# Lay mo hinh tot nhat
best_model = grid_search.best_estimator_

# Du doan tren tap test
y_pred = best_model.predict(X_test)

# Tinh accuracy
test_accuracy = accuracy_score(y_test, y_pred)

print("Best parameters:", grid_search.best_params_)
print(f"Cross-validation accuracy: {grid_search.best_score_ * 100:.2f}%")
print(f"Test accuracy: {test_accuracy * 100:.2f}%")

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=iris.target_names
))


In [ ]:

import pandas as pd

# Lay ket qua GridSearchCV
cv_results = pd.DataFrame(grid_search.cv_results_)

# Chon cac cot quan trong
result_table = cv_results[[
    'params',
    'mean_test_score',
    'std_test_score',
    'rank_test_score'
]]

# Sap xep tu tot nhat den thap nhat
result_table = result_table.sort_values(
    by='rank_test_score'
)

display(result_table.head(10))


In [ ]:

import matplotlib.pyplot as plt
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

# Du doan bang mo hinh tot nhat
y_pred = best_model.predict(X_test)

# Tinh ma tran nham lan
cm = confusion_matrix(y_test, y_pred)

print("Confusion Matrix:")
print(cm)

# Ve heatmap
fig, ax = plt.subplots(figsize=(7, 5))

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=iris.target_names
)

disp.plot(cmap="Blues", ax=ax, values_format="d")

plt.title("Confusion Matrix - Iris SVM")
plt.tight_layout()

# Luu hinh
plt.savefig(
    "/kaggle/working/iris_confusion_matrix.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


In [ ]:

from sklearn.metrics import (
    accuracy_score,
    classification_report
)

# Danh gia mo hinh
accuracy = accuracy_score(y_test, y_pred)

print("===== SVM IRIS EVALUATION =====")
print("Best Parameters:", grid_search.best_params_)
print(f"CV Accuracy: {grid_search.best_score_ * 100:.2f}%")
print(f"Test Accuracy: {accuracy * 100:.2f}%")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        y_pred,
        target_names=iris.target_names
    )
)


In [ ]:

from sklearn.metrics import classification_report

report = classification_report(
    y_test,
    y_pred,
    target_names=iris.target_names
)

with open(
    "/kaggle/working/iris_evaluation.txt",
    "w",
    encoding="utf-8"
) as f:
    f.write("SVM IRIS EVALUATION\n\n")
    f.write(f"Best Parameters: {grid_search.best_params_}\n")
    f.write(f"CV Accuracy: {grid_search.best_score_:.4f}\n")
    f.write(f"Test Accuracy: {accuracy:.4f}\n\n")
    f.write(report)

print("Da luu bao cao thanh cong!")


### Kết luận bài thực hành SVM trên Iris

Bài thực hành xây dựng mô hình Support Vector Machine (SVM) để phân loại 3 loài hoa Iris với 150 mẫu dữ liệu và 4 đặc trưng.

Dữ liệu được chia theo tỷ lệ 80% huấn luyện và 20% kiểm tra, đồng thời được chuẩn hóa bằng StandardScaler.

Kết quả thử nghiệm bốn kernel:

| Kernel | Test Accuracy |
|---|---|
| Linear | 100% |
| Polynomial | 90% |
| RBF | 96,67% |
| Sigmoid | 100% |

Sử dụng GridSearchCV kết hợp 5-fold Cross-Validation, mô hình được lựa chọn có các tham số tối ưu là **kernel = linear, C = 1**, với độ chính xác trung bình qua Cross-Validation đạt **97,50%**.

Khi đánh giá trên 30 mẫu kiểm tra, mô hình đạt Accuracy **100%**. Ma trận nhầm lẫn cho thấy toàn bộ 30 mẫu được phân loại đúng, không có trường hợp nhầm lẫn giữa ba loài hoa.

**Kết luận:** SVM với kernel linear cho kết quả phân loại rất tốt trên tập dữ liệu Iris trong thực nghiệm này. Tuy nhiên, kết quả 100% chỉ phản ánh tập kiểm tra hiện tại và không đảm bảo mô hình luôn dự đoán chính xác tuyệt đối trên dữ liệu mới.